# Fine-tuning Pipeline (MACE-MH-1 OMAT)

This notebook uses multi-head tuning with 2 datasets:
    * DFT relaxation set constructed below
    * configs from OMAT head w/ energy + force labels from MACE-MH-1 predictions (to preserve original model behavior)
      replay source [06/23/26]: https://github.com/ACEsuit/mace-foundations/releases/tag/mace_mh_1

See https://mace-docs.readthedocs.io/en/latest/guide/multihead_finetuning.html for more details.

## Conda Env Setup
In the CLI, run 
`path/mace-mh-1.yml`

In [2]:
from ase.io import iread, write

In [ ]:
# ===== add DFT runs here =====

base_path = "data/dft-data/"

vasprun_paths = (
        []
        )

In [1]:
# create finetune set by storing the first and last frames of each DFT run as an ASE Atoms object

def load_frames(path):
    # return tuple of first and last frames of DFT run 
    frames = iread(path)
    first = last = next(frames)
    for f in frames:
        last = f

    return first, last

In [ ]:
# create potential finetune sets by storing the first and last frames of each DFT run as an ASE Atoms object in train.xyz

all_frames = []
for path in vasprun_paths:
    first, last = load_frames(path)
    all_frames.extend([first, last])

**Create the combined finetuning set** with the following CLI command.
This calls fine_tuning_select.py (stored in the mace-torch package you installed in your conda env) 
to create a filtered 3rd file from your dft data and the replay. Edit flags below filepaths at will.

```bash 
python -m mace.cli.fine_tuning_select \
  --configs_pt data/replay-data-mh-1-omat-pbe.xyz \
  --configs_ft data/train.xyz \
  --num_samples 10000 \
  --subselect fps \
  --model path/to/foundation_model.model \
  --output selected_configs.xyz \
  --filtering_type combinations \
  --head_pt pt_head \
  --head_ft target_head \
  --weight_pt 1.0 \
  --weight_ft 10.0
```
Parameters:
    * `--configs_pt`: Path to the replay dataset
    * `--configs_ft`: Path to your target dataset
    * `--num_samples`: Number of configurations to select from the replay dataset
    * `--subselect`: Method for subselection (fps for Farthest Point Sampling or random)
    * `--filtering_type`: How to filter configurations based on elements:
        * `combinations`: Keep configurations with combinations of elements in your target dataset
        * `exclusive`: Keep configurations containing only elements in your target dataset
        * `inclusive`: Keep configurations containing all elements in your target dataset
        * `none`: No filtering
    * `--atomic_numbers`: Optionally specify specific atomic numbers to filter by

# FIX PATHS # **Train model on combined set** with the following CLI command.

```bash
python -m mace.cli.run_train \
  --name myMACE_finetuned \
  --pt_train_file selected_configs.xyz \
  --train_file path/to/your_dataset.xyz \
  --valid_fraction 0.05 \
  --foundation_model path/to/foundation_model.model \
  --energy_weight 1.0 \
  --forces_weight 100.0 \
  --swa \
  --swa_energy_weight 10.0 \
  --swa_forces_weight 100.0
```

In [ ]:
from mace.calculators import MACECalculator

mace_calc = MACECalculator(
        model_paths="",
        device="mps",
        default_dtype="float32"
        )